In [ ]:
import os
from glob import glob
import numpy as np
from PIL import Image

import tensorflow as tf

# ---------------- GPU setup ----------------
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Using GPU(s): {gpus}")
    except RuntimeError as e:
        print("Could not set memory growth:", e)
else:
    print("⚠️ No GPU found. Running on CPU.")

# Optional: XLA
tf.config.optimizer.set_jit(True)

# Optional: Mixed precision (recommended on RTX GPUs)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

from huggingface_hub import hf_hub_download

# ============================================================
# 0) CONFIG
# ============================================================
REPO_ID = "google/maxim-s2-dehazing-sots-outdoor"
DATA_ROOT = r".\data"

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 10
LR = 2e-4

SAVEDMODEL_DIR = os.path.abspath("./maxim_savedmodel")
ADAPTER_BEST_WEIGHTS = os.path.abspath("./adapter_best.weights.h5")

AUTOTUNE = tf.data.AUTOTUNE
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff")


# ============================================================
# 1) Download SavedModel
# ============================================================
def download_savedmodel(repo_id: str, out_dir: str):
    os.makedirs(os.path.join(out_dir, "variables"), exist_ok=True)
    files = [
        "saved_model.pb",
        "keras_metadata.pb",
        "variables/variables.index",
        "variables/variables.data-00000-of-00001",
    ]
    for f in files:
        hf_hub_download(
            repo_id=repo_id,
            filename=f,
            local_dir=out_dir,
            local_dir_use_symlinks=False,
        )
    return out_dir

print("Downloading:", REPO_ID)
download_savedmodel(REPO_ID, SAVEDMODEL_DIR)

print("Loading SavedModel backbone...")
base_model = tf.keras.models.load_model(SAVEDMODEL_DIR, compile=False)
base_model.trainable = False  # frozen


# ============================================================
# 2) Output selection utils
# ============================================================
def flatten_tensors(y):
    out = []
    if isinstance(y, tf.Tensor):
        out.append(y)
    elif isinstance(y, (list, tuple)):
        for v in y:
            out.extend(flatten_tensors(v))
    elif isinstance(y, dict):
        for v in y.values():
            out.extend(flatten_tensors(v))
    return out

def select_best_output(y, prefer_hw=None):
    tensors = [t for t in flatten_tensors(y) if isinstance(t, tf.Tensor) and len(t.shape) == 4]
    if not tensors:
        raise ValueError("No 4D tensors found in backbone outputs.")

    if prefer_hw is not None:
        H, W = prefer_hw
        same = [t for t in tensors if (t.shape[1] == H and t.shape[2] == W)]
        if same:
            return same[-1]

    def area(t):
        h = t.shape[1] if t.shape[1] is not None else 0
        w = t.shape[2] if t.shape[2] is not None else 0
        return int(h) * int(w)

    return max(tensors, key=area)

@tf.function
def backbone_predict(x):
    y = base_model(x, training=False)
    pred = select_best_output(y, prefer_hw=(IMG_SIZE, IMG_SIZE))
    pred = tf.image.resize(pred, (IMG_SIZE, IMG_SIZE), method="bilinear")
    pred = tf.clip_by_value(pred, 0.0, 1.0)
    return pred

_tmp = backbone_predict(tf.random.uniform([1, IMG_SIZE, IMG_SIZE, 3]))
print("Backbone prediction shape:", _tmp.shape)


# ============================================================
# 3) Adapter head (trainable)
# ============================================================
adapter = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(3, 3, padding="same", activation=None),
], name="adapter_head")

optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

def restoration_loss(y_true, y_pred):
    # Force loss math in float32 for stability under mixed precision
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = tf.reduce_mean(1.0 - tf.image.ssim(y_true, y_pred, max_val=1.0))
    return l1 + 0.2 * ssim


# ============================================================
# 4) Dataset: pair by stem
# ============================================================
def list_images(folder):
    paths = []
    for ext in IMG_EXTS:
        paths.extend(glob(os.path.join(folder, f"*{ext}")))
        paths.extend(glob(os.path.join(folder, f"*{ext.upper()}")))
    return sorted(paths)

def build_pairs(input_dir, target_dir):
    in_paths = list_images(input_dir)
    tg_paths = list_images(target_dir)
    tg_map = {os.path.splitext(os.path.basename(p))[0]: p for p in tg_paths}

    pairs, missing = [], []
    for p in in_paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        if stem in tg_map:
            pairs.append((p, tg_map[stem]))
        else:
            missing.append(stem)

    print(f"[PAIRING] {os.path.basename(os.path.dirname(input_dir))} "
          f"Inputs: {len(in_paths)} | Targets: {len(tg_paths)} | Paired: {len(pairs)}")
    if missing:
        print("[PAIRING] Missing targets (first 10):", missing[:10])
    if not pairs:
        raise ValueError("No pairs found. Check folders/naming.")
    return pairs

def _load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img

def _augment(x, y):
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x)
        y = tf.image.flip_left_right(y)
    return x, y

def make_dataset(split):
    in_dir = os.path.join(DATA_ROOT, split, "input")
    tg_dir = os.path.join(DATA_ROOT, split, "target")

    pairs = build_pairs(in_dir, tg_dir)
    in_list, tg_list = zip(*pairs)

    ds = tf.data.Dataset.from_tensor_slices((list(in_list), list(tg_list)))

    def _map(in_p, tg_p):
        return _load_image(in_p), _load_image(tg_p)

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE).shuffle(512, reshuffle_each_iteration=True)

    # Extra TF data performance hints
    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset("train")
val_ds = make_dataset("val")

xb, yb = next(iter(train_ds))
bp = backbone_predict(xb)
print("Sanity - xb:", xb.shape, "yb:", yb.shape, "backbone_pred:", bp.shape)


# ============================================================
# 5) Custom training loop
# ============================================================
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        base_pred = backbone_predict(x)
        delta = adapter(base_pred, training=True)
        out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)
        loss = restoration_loss(y, out)

    grads = tape.gradient(loss, adapter.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(grads, adapter.trainable_variables) if g is not None]
    optimizer.apply_gradients(grads_and_vars)
    return loss

@tf.function
def val_step(x, y):
    base_pred = backbone_predict(x)
    delta = adapter(base_pred, training=False)
    out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)
    return restoration_loss(y, out)

best_val = float("inf")

print("\nTraining adapter head (GPU if available)...")
for epoch in range(1, EPOCHS + 1):
    train_losses = []
    for x, y in train_ds:
        loss = train_step(x, y)
        train_losses.append(loss)
    train_loss = tf.reduce_mean(train_losses).numpy().item()

    val_losses = []
    for x, y in val_ds:
        vloss = val_step(x, y)
        val_losses.append(vloss)
    val_loss = tf.reduce_mean(val_losses).numpy().item()

    print(f"Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        adapter.save_weights(ADAPTER_BEST_WEIGHTS)
        print("  ✔ Saved best adapter weights:", ADAPTER_BEST_WEIGHTS)

print("Done. Best val_loss:", best_val)


# ============================================================
# 6) Inference
# ============================================================
def enhance_image(input_path, output_path, weights_path=ADAPTER_BEST_WEIGHTS):
    adapter.load_weights(weights_path)

    img = Image.open(input_path).convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    x = tf.convert_to_tensor(arr)[None, ...]
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))

    base_pred = backbone_predict(x)
    delta = adapter(base_pred, training=False)
    out = tf.clip_by_value(base_pred + delta, 0.0, 1.0)[0].numpy()

    Image.fromarray((out * 255).astype(np.uint8)).save(output_path)
    print("Wrote:", output_path)

# Example:
# enhance_image(r".\test_input.jpg", r".\test_output.jpg")
